In [1]:
import pandas as pd
import ast
import numpy as np
import random

In [2]:
training_file = pd.read_pickle("/nfs/primary/sas_reranker/d2qmm_ohsumed_training_data.pkl")

In [20]:
training_file[training_file["medline_ui"] == 88077220]

,medline_ui,title,abstract,sensitivity,text,querygen,querygen_score,best_query
8881,88077220,A comparison of changes seen on radiographs of...,To test the hypothesis that rheumatoid arthrit...,0,A comparison of changes seen on radiographs of...,what is the severity of rheumatoid arthritis?\...,"[0.5374905, 1.4450203, 2.4346664, 4.031351, 1....",is it milder to have rheumatoid arthritis in c...


In [21]:
def get_max_score_query(row):
    try:
        queries = row['querygen'].split('\n') if row['querygen'] else []  # Split queries by newline
        scores = row['querygen_score']  # Use the array directly
        max_index = np.argmax(scores)  # Find the index of the max score
        return queries[max_index]  # Return the query with the highest score
    except ValueError:
        pass

# Apply the function to the dataframe
training_file['best_query'] = training_file.apply(get_max_score_query, axis=1)

In [22]:
removed = training_file[training_file.best_query.isna()].medline_ui.tolist()
# with open('removed_docnos.txt', 'w+') as f:
    
#     # write elements of list
#     for items in removed:
#         f.write('%s\n' %items)

88077220 in removed

False

In [24]:
training_file[training_file["medline_ui"] == 88077220]

,medline_ui,title,abstract,sensitivity,text,querygen,querygen_score,best_query
8881,88077220,A comparison of changes seen on radiographs of...,To test the hypothesis that rheumatoid arthrit...,0,A comparison of changes seen on radiographs of...,what is the severity of rheumatoid arthritis?\...,"[0.5374905, 1.4450203, 2.4346664, 4.031351, 1....",is it milder to have rheumatoid arthritis in c...


In [ ]:
88077220

In [8]:
training_file['qid'] = training_file.index
training_file.to_csv("./data/training_data_w_qids.csv", index = False)

In [9]:
training_file

,medline_ui,title,abstract,sensitivity,text,querygen,querygen_score,best_query,qid
0,88000001,The binding of acetaldehyde to the active site...,"Ribonuclease A was reacted with [1-13C,1,2-14C...",0,The binding of acetaldehyde to the active site...,what site of activity is used to bind ribonucl...,"[0.88141286, 2.3967025, 0.32945463, 0.9057854,...",does acetaldehyde react with ribonuclease A,0
1,88000002,Reductions in breath ethanol readings in norma...,Blood ethanol concentrations were measured seq...,0,Reductions in breath ethanol readings in norma...,is ethanol in breath normal\nis blood ethanol ...,"[2.034942, 3.5010455, 0.3425358, 0.5842041, 0....",is blood ethanol measured sequentially for a n...,1
2,88000003,Does the blockade of opioid receptors influenc...,We have tested whether the opioid antagonists ...,0,Does the blockade of opioid receptors influenc...,which opioids can cause ethanol dependence?\ne...,"[2.229129, 1.2757981, 2.7790852, 0.18533784, 2...",does the blockade of opioid receptors influenc...,2
3,88000006,Drinkwatchers--description of subjects and eva...,Clinical examination and measurement of MCV an...,0,Drinkwatchers--description of subjects and eva...,heavy drinking blood test\nhow much alcohol is...,"[0.87555873, 0.48380333, 1.1801778, 0.89115256...",how much alcohol do drinkwatchers drink,3
4,88000007,Platelet affinity for serotonin is increased i...,The kinetics of 3H serotonin platelet uptake w...,0,Platelet affinity for serotonin is increased i...,what is the biological marker for dependence\n...,"[3.0765808, 1.4992186, 1.945047, 1.81894, 2.75...",does alcoholics use serotonin to increase plat...,4
...,...,...,...,...,...,...,...,...,...
334129,87325749,Clinical investigation of Duchenne muscular dy...,Between 1979 and 1987 we documented the natura...,0,Clinical investigation of Duchenne muscular dy...,what is clinical investigation of duchenne mus...,"[4.7701135, 3.1781783, 3.1509128, 3.3405652, 3...",what is clinical investigation of duchenne mus...,334129
334130,87325791,Prostaglandins. Old concepts and new perspecti...,Lupus anticoagulants and anticardiolipin antib...,0,Prostaglandins. Old concepts and new perspecti...,what prostaglandins do\nprostaglandins lupus\n...,"[0.8321806, 1.5327109, 3.6516025, 2.132977, 0....",what is a lupus anticoagulant and anticardioli...,334130
334131,87325811,Vinculin in focal cell-to-substrate attachment...,Vinculin is a specialized protein with a molec...,0,Vinculin in focal cell-to-substrate attachment...,what is the function of vinculin\nwhat is vinc...,"[1.5229188, 3.623153, 3.2570174, 2.759778, 1.5...",where does vinculin attach in cells to a subst...,334131
334132,87325849,Intraocular fibrinolysis with recombinant huma...,Intraocular fibrin formation following ocular ...,0,Intraocular fibrinolysis with recombinant huma...,recombinant plasminogen activator treatment\ni...,"[1.6216362, 3.4744203, 3.2590685, 1.8076218, 0...",intraocular fibrinolysis with recombinant plas...,334132


In [26]:
with open("./data/training_queries.tsv", "w") as file:
    for _, row in training_file.iterrows():
        file.write(f"{_}\t{row.best_query}\n")

In [27]:
query_df = pd.read_csv("./data/training_queries.tsv", sep='\t', names = ["qid", "query"])
query_df[query_df["query"].isna()]

,qid,query


In [29]:
# Get top 100 BM25 docs
import pandas as pd
import os
from pyterrier_pisa import PisaIndex
import pyterrier as pt
if not pt.started():
    pt.init()

# In[17]:


def get_bm25(
    query_file: str,
    output_dir: str,
    index_path: str,
    depth: int = 100,
    threads: int = 4):
    index = PisaIndex(index_path, threads = threads)
    model = index.bm25(verbose = True) % depth
    query_df = pd.read_csv(query_file, sep='\t', names = ["qid", "query"])
    query_df = query_df
    result = model.transform(query_df)        
    out_file = f"{output_dir}/bm25.{depth}.rez.gz"
    pt.io.write_results(result, out_file)

    return f"Successfully ran BM25 on {query_file} and saved to {out_file}"

get_bm25(query_file = "./data/training_queries.tsv", output_dir = "./data", index_path = "/nfs/primary/scratch/classifier_scoring/pisa_index_v2", threads = 1024)

/tmp/ipykernel_18383/777426493.py:6: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
PISA bm25: 100%|██████████| 334070/334070 [00:28<00:00, 11699.49query/s]


'Successfully ran BM25 on ./data/training_queries.tsv and saved to ./data/bm25.100.rez.gz'

In [31]:
result_df = pt.io.read_results("./data/bm25.100.rez.gz")

In [38]:
qid_to_docnos = result_df.groupby("qid")["docno"].apply(list).to_dict()
index_to_docno = training_file['medline_ui'].to_dict()

In [68]:
with open("./data/bm25_7neg_triples.tsv", "w") as f:
    for qid in qid_to_docnos:
        positive = index_to_docno[int(qid)]
        candidates = [int(doc) for doc in qid_to_docnos[qid] if int(doc) != int(positive)]
        if len(candidates) >= 7:
            negatives = random.sample(candidates, 7)
            for negative in negatives:
                f.write(f"{qid}\t{positive}\t{negative}\n")